# 02. Gradient Descent: El Corazón del Aprendizaje Automático

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 120 minutos (expandido con ejercicios)  
**Prerequisitos:** [01. Regresión Lineal](01-regresion-lineal.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender cómo el gradient descent encuentra mínimos de funciones
- **Implementar desde cero** las tres variantes: Batch, Stochastic y Mini-batch GD
- Comprender el impacto crítico del learning rate
- **Aplicar técnicas de optimización avanzadas** (Momentum, RMSprop, Adam)
- Visualizar la superficie de pérdida y el camino de optimización
- **Completar 8 ejercicios prácticos autogradeados** (100 puntos totales)
- Diagnosticar y solucionar problemas comunes de convergencia

<a name='toc'></a>
## 📚 Tabla de Contenidos

- [1 - 📌 Motivación: ¿Por qué Gradient Descent?](#1)
- [2 - 📊 Intuición Visual: Descendiendo por una Montaña](#2)
- [3 - 🧮 Fundamentos Matemáticos](#3)
- [4 - 💻 Implementación Desde Cero](#4)
- [**5 - 🎓 Ejercicios Prácticos Guiados (100 pts)**](#5)
  - [Exercise 1 - `compute_gradient` (10 pts)](#ex-1)
  - [Exercise 2 - `gradient_descent_step` (10 pts)](#ex-2)
  - [Exercise 3 - `batch_gradient_descent` (15 pts)](#ex-3)
  - [Exercise 4 - `create_mini_batches` (10 pts)](#ex-4)
  - [Exercise 5 - `sgd_with_momentum` (15 pts)](#ex-5)
  - [Exercise 6 - `rmsprop_update` (15 pts)](#ex-6)
  - [Exercise 7 - `adam_optimizer` (20 pts)](#ex-7)
  - [Exercise 8 - `learning_rate_decay` (5 pts)](#ex-8)
- [6 - 🏭 Comparación con Frameworks](#6)
- [7 - 🔬 Ejercicios Avanzados](#7)
- [**8 - 📄 Papers y Referencias (15+ papers)**](#8)
- [9 - 📚 Resumen](#9)
- [10 - ➡️ Navegación](#10)

---

**⚡ Nuevo en esta versión:**
- ✅ 8 ejercicios autogradeados estilo Coursera
- ✅ Sistema de puntos (100 pts totales, 70 pts para aprobar)
- ✅ Sección de papers con 15+ referencias específicas
- ✅ Ejercicios avanzados opcionales

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades personalizadas
import sys
sys.path.append('../../shared/utils')
from visualization import plot_loss_surface, plot_optimization_path
from testing import test_exercise, check_shape, check_close
from datasets import load_dataset, generate_synthetic_regression

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---
## 📌 1. Motivación: ¿Por qué Gradient Descent?

### El Problema del Mundo Real

Imagina que estás perdido en las montañas durante una noche de niebla densa. Tu objetivo es llegar al valle (el punto más bajo). No puedes ver el paisaje completo, solo puedes:

1. Sentir la pendiente bajo tus pies
2. Dar un paso en la dirección más empinada hacia abajo
3. Repetir hasta llegar al fondo

**¡Esto es exactamente lo que hace Gradient Descent!**

### ¿Por qué es tan importante?

- 🧠 **Es el algoritmo de optimización fundamental** usado en casi todos los modelos de ML
- 🔥 **Entrena las redes neuronales** modernas (GPT, DALL-E, etc.)
- 📈 **Escala a millones de parámetros** (la Normal Equation no puede)
- 🎯 **Es elegante y general** - funciona para cualquier función diferenciable

### Aplicaciones Reales

- 🤖 Entrenamiento de redes neuronales profundas
- 📊 Optimización de modelos de ML en general
- 🎮 Entrenamiento de agentes de Reinforcement Learning
- 💰 Optimización de portafolios financieros
- 🔬 Ajuste de modelos científicos complejos

### La Pregunta Guía

> **¿Cómo puede un algoritmo encontrar el mínimo de una función sin ver toda la superficie, y qué factores determinan su éxito?**

---
## 📊 2. Intuición Visual: Descendiendo por una Montaña

Empecemos visualizando qué es lo que queremos optimizar.

In [ ]:
# Función cuadrática simple en 1D para visualización
def f(x):
    """Función f(x) = (x - 3)^2 + 1"""
    return (x - 3)**2 + 1

def df(x):
    """Derivada: f'(x) = 2(x - 3)"""
    return 2 * (x - 3)

# Crear la curva
x_vals = np.linspace(-2, 8, 200)
y_vals = f(x_vals)

# Simular gradient descent
def gradient_descent_1d(start, learning_rate, n_iterations):
    """Ejecuta gradient descent en 1D"""
    path = [start]
    x = start
    
    for i in range(n_iterations):
        gradient = df(x)
        x = x - learning_rate * gradient
        path.append(x)
    
    return np.array(path)

# Ejecutar con diferentes learning rates
paths = {
    'Muy pequeño (lr=0.05)': gradient_descent_1d(start=7.0, learning_rate=0.05, n_iterations=50),
    'Óptimo (lr=0.3)': gradient_descent_1d(start=7.0, learning_rate=0.3, n_iterations=20),
    'Muy grande (lr=0.9)': gradient_descent_1d(start=7.0, learning_rate=0.9, n_iterations=30)
}

# Visualización
fig = go.Figure()

# Función objetivo
fig.add_trace(go.Scatter(
    x=x_vals,
    y=y_vals,
    mode='lines',
    name='Función f(x) = (x-3)² + 1',
    line=dict(color='lightblue', width=3)
))

# Mínimo verdadero
fig.add_trace(go.Scatter(
    x=[3],
    y=[1],
    mode='markers',
    name='Mínimo Global',
    marker=dict(color='red', size=15, symbol='star')
))

# Paths de optimización
colors = ['orange', 'green', 'purple']
for (name, path), color in zip(paths.items(), colors):
    fig.add_trace(go.Scatter(
        x=path,
        y=f(path),
        mode='lines+markers',
        name=name,
        line=dict(color=color, width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title="Gradient Descent con Diferentes Learning Rates",
    xaxis_title="Parámetro x",
    yaxis_title="Costo f(x)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Observaciones Clave:")
print("   • Learning rate pequeño: Progreso lento pero estable")
print("   • Learning rate óptimo: Convergencia rápida y suave")
print("   • Learning rate grande: Oscila y puede divergir")
print("   • El gradiente (pendiente) indica la dirección de mayor aumento")
print("   • Nos movemos en dirección OPUESTA al gradiente (descendiendo)")

In [ ]:
# Visualización 3D: Superficie de pérdida para regresión lineal
# Generar datos simples
np.random.seed(42)
X_simple = 2 * np.random.rand(50, 1)
y_simple = 4 + 3 * X_simple.flatten() + np.random.randn(50)

# Crear grid de parámetros
w_range = np.linspace(0, 6, 50)
b_range = np.linspace(-2, 8, 50)
W, B = np.meshgrid(w_range, b_range)

# Calcular MSE para cada combinación de w y b
MSE = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w, b = W[i, j], B[i, j]
        y_pred = w * X_simple.flatten() + b
        MSE[i, j] = np.mean((y_simple - y_pred) ** 2)

# Visualización 3D
fig = go.Figure(data=[go.Surface(
    z=MSE,
    x=w_range,
    y=b_range,
    colorscale='Viridis',
    name='Superficie de Pérdida'
)])

fig.update_layout(
    title="Superficie de Pérdida MSE (2 Parámetros)",
    scene=dict(
        xaxis_title='Weight (w)',
        yaxis_title='Bias (b)',
        zaxis_title='MSE',
    ),
    template="plotly_white",
    height=600
)

fig.show()

print("\n💡 Esta es la 'montaña' que Gradient Descent desciende.")
print("   • El punto más bajo es la combinación óptima de w y b")
print("   • El algoritmo solo puede ver la pendiente local, no toda la superficie")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $\theta$ | Parámetros del modelo (puede ser vector) |
| $J(\theta)$ | Función de costo (pérdida) |
| $\nabla_\theta J$ | Gradiente de J respecto a $\theta$ |
| $\alpha$ | Learning rate (tasa de aprendizaje) |
| $t$ | Iteración actual |
| $m$ | Número de ejemplos de entrenamiento |

### El Algoritmo de Gradient Descent

**Idea Central:** Actualizar los parámetros en la dirección opuesta al gradiente.

$$
\begin{align}
\theta^{(t+1)} &= \theta^{(t)} - \alpha \nabla_\theta J(\theta^{(t)}) \tag{1}
\end{align}
$$

Donde:
- $\theta^{(t)}$ son los parámetros en la iteración $t$
- $\alpha$ controla el tamaño del paso
- $\nabla_\theta J$ es el vector de derivadas parciales

### ¿Por qué funciona?

**Teorema de Taylor (aproximación de primer orden):**

$$
J(\theta + \Delta\theta) \approx J(\theta) + \nabla_\theta J \cdot \Delta\theta \tag{2}
$$

Para minimizar $J$, queremos que $J(\theta + \Delta\theta) < J(\theta)$.

Si elegimos $\Delta\theta = -\alpha \nabla_\theta J$:

$$
\begin{align}
J(\theta - \alpha \nabla_\theta J) &\approx J(\theta) - \alpha \|\nabla_\theta J\|^2 \tag{3}\\
&< J(\theta) \quad \text{(si } \alpha \text{ es pequeño)} \tag{4}
\end{align}
$$

**Conclusión:** Moverse en dirección opuesta al gradiente ¡garantiza reducir la pérdida!

### Derivadas para Regresión Lineal

Recordemos la función de costo MSE:

$$
J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2 = \frac{1}{m} \sum_{i=1}^{m} (w^T x^{(i)} + b - y^{(i)})^2 \tag{5}
$$

Las derivadas parciales son:

$$
\begin{align}
\frac{\partial J}{\partial w_j} &= \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x_j^{(i)} \tag{6}\\
\frac{\partial J}{\partial b} &= \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \tag{7}
\end{align}
$$

**Nota:** El factor 2 se suele omitir (solo escala el learning rate).

### Ejemplo Numérico

Supongamos:
- 3 ejemplos: $(x, y) = \{(1, 3), (2, 5), (3, 7)\}$
- Parámetros actuales: $w = 1.5$, $b = 1.0$
- Learning rate: $\alpha = 0.1$

**Paso 1: Calcular predicciones**
$$
\begin{align}
\hat{y}^{(1)} &= 1.5(1) + 1.0 = 2.5 \\
\hat{y}^{(2)} &= 1.5(2) + 1.0 = 4.0 \\
\hat{y}^{(3)} &= 1.5(3) + 1.0 = 5.5
\end{align}
$$

**Paso 2: Calcular errores**
$$
\begin{align}
e^{(1)} &= 2.5 - 3 = -0.5 \\
e^{(2)} &= 4.0 - 5 = -1.0 \\
e^{(3)} &= 5.5 - 7 = -1.5
\end{align}
$$

**Paso 3: Calcular gradientes**
$$
\begin{align}
\frac{\partial J}{\partial w} &= \frac{1}{3}[(-0.5)(1) + (-1.0)(2) + (-1.5)(3)] = \frac{-7.0}{3} \approx -2.33 \\
\frac{\partial J}{\partial b} &= \frac{1}{3}[-0.5 - 1.0 - 1.5] = -1.0
\end{align}
$$

**Paso 4: Actualizar parámetros**
$$
\begin{align}
w_{new} &= 1.5 - 0.1(-2.33) = 1.5 + 0.233 = 1.733 \\
b_{new} &= 1.0 - 0.1(-1.0) = 1.0 + 0.1 = 1.1
\end{align}
$$

In [ ]:
# Verificación del ejemplo numérico
X_example = np.array([1, 2, 3])
y_example = np.array([3, 5, 7])
w, b = 1.5, 1.0
alpha = 0.1

# Paso 1: Predicciones
y_pred = w * X_example + b
print("Predicciones:", y_pred)

# Paso 2: Errores
errors = y_pred - y_example
print("Errores:", errors)

# Paso 3: Gradientes
dw = np.mean(errors * X_example)
db = np.mean(errors)
print(f"Gradiente w: {dw:.3f}")
print(f"Gradiente b: {db:.3f}")

# Paso 4: Actualización
w_new = w - alpha * dw
b_new = b - alpha * db
print(f"\nNuevos parámetros:")
print(f"w: {w:.3f} → {w_new:.3f}")
print(f"b: {b:.3f} → {b_new:.3f}")

### Las Tres Variantes de Gradient Descent

#### 1. Batch Gradient Descent

Usa **todos** los datos en cada iteración:

$$
\theta := \theta - \alpha \frac{1}{m} \sum_{i=1}^{m} \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{8}
$$

✅ **Ventajas:** Convergencia suave, dirección exacta del gradiente  
❌ **Desventajas:** Lento para datasets grandes

#### 2. Stochastic Gradient Descent (SGD)

Usa **un** ejemplo aleatorio por iteración:

$$
\theta := \theta - \alpha \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{9}
$$

✅ **Ventajas:** Muy rápido, puede escapar mínimos locales  
❌ **Desventajas:** Convergencia ruidosa, nunca se estabiliza completamente

#### 3. Mini-batch Gradient Descent

Usa **un subconjunto** (batch) de ejemplos:

$$
\theta := \theta - \alpha \frac{1}{B} \sum_{i=k}^{k+B-1} \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{10}
$$

Donde $B$ es el tamaño del batch (típicamente 32, 64, 128, 256).

✅ **Ventajas:** Balance entre velocidad y estabilidad, aprovecha hardware (GPU)  
❌ **Desventajas:** Requiere tunear el batch size

**Comparación:**

| Variante | Ejemplos/iter | Velocidad | Convergencia | Uso |
|----------|---------------|-----------|--------------|-----|
| Batch | $m$ (todos) | Lenta | Suave | Datasets pequeños |
| SGD | 1 | Muy rápida | Ruidosa | Algoritmos online |
| Mini-batch | $B$ (típ. 32-256) | Rápida | Balanceada | **Más usado** |

---
## 💻 4. Implementación Desde Cero

In [ ]:
class GradientDescentOptimizer:
    """
    Implementación completa de Gradient Descent con sus variantes.
    
    Implementa:
    - Batch Gradient Descent
    - Stochastic Gradient Descent (SGD)
    - Mini-batch Gradient Descent
    - Optimizadores avanzados: Momentum, RMSprop, Adam
    
    Parameters:
    -----------
    learning_rate : float
        Tasa de aprendizaje
    n_iterations : int
        Número de iteraciones (epochs)
    batch_size : int or None
        Tamaño del batch. None = Batch GD, 1 = SGD, otro = Mini-batch
    optimizer : str
        'gd', 'momentum', 'rmsprop', 'adam'
    momentum : float
        Parámetro de momentum (beta1)
    beta2 : float
        Parámetro para RMSprop/Adam
    epsilon : float
        Término pequeño para estabilidad numérica
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=None,
                 optimizer='gd', momentum=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.batch_size = batch_size
        self.optimizer = optimizer
        self.momentum = momentum
        self.beta2 = beta2
        self.epsilon = epsilon
        
        # Parámetros del modelo
        self.weights = None
        self.bias = None
        
        # Para tracking
        self.losses = []
        self.weight_history = []  # Para visualizar el camino
        
        # Para optimizadores con momento
        self.v_w = None  # Velocidad para weights (momentum)
        self.v_b = None  # Velocidad para bias
        self.s_w = None  # Segundo momento para weights (RMSprop/Adam)
        self.s_b = None  # Segundo momento para bias
        self.t = 0       # Contador de tiempo para Adam
    
    def fit(self, X, y):
        """
        Entrena el modelo usando gradient descent.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features de entrenamiento
        y : np.ndarray, shape (n_samples,)
            Target values
        """
        # Asegurar formato correcto
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        # Inicialización de parámetros (pequeños valores aleatorios)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        # Inicializar momentos
        self.v_w = np.zeros_like(self.weights)
        self.v_b = 0.0
        self.s_w = np.zeros_like(self.weights)
        self.s_b = 0.0
        
        # Determinar batch size
        if self.batch_size is None:
            batch_size = n_samples  # Batch GD
        else:
            batch_size = min(self.batch_size, n_samples)
        
        print(f"\n🏃 Iniciando entrenamiento...")
        print(f"   Optimizador: {self.optimizer}")
        print(f"   Batch size: {batch_size} {'(Batch GD)' if batch_size == n_samples else '(Mini-batch)' if batch_size > 1 else '(SGD)'}")
        print(f"   Learning rate: {self.lr}")
        print()
        
        # Training loop
        for epoch in range(self.n_iter):
            # Shuffle de datos (importante para SGD/Mini-batch)
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # Procesar en batches
            epoch_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, batch_size):
                # Obtener batch
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                batch_samples = len(X_batch)
                
                # Forward pass
                y_pred = X_batch @ self.weights + self.bias
                
                # Calcular loss
                loss = np.mean((y_batch - y_pred) ** 2)
                epoch_loss += loss
                n_batches += 1
                
                # Calcular gradientes
                error = y_pred - y_batch
                dw = (1 / batch_samples) * (X_batch.T @ error)
                db = (1 / batch_samples) * np.sum(error)
                
                # Actualizar parámetros según el optimizador
                self._update_parameters(dw, db)
            
            # Promedio de loss de la época
            avg_loss = epoch_loss / n_batches
            self.losses.append(avg_loss)
            
            # Guardar historial de weights para visualización
            if n_features <= 2:  # Solo para visualización 2D/3D
                self.weight_history.append((self.weights.copy(), self.bias))
            
            # Logging
            if epoch % max(1, self.n_iter // 10) == 0:
                print(f"Epoch {epoch:4d} - Loss: {avg_loss:.6f}")
        
        print(f"\n✅ Entrenamiento completado")
        print(f"   Loss final: {self.losses[-1]:.6f}")
        print(f"   Weights: {self.weights}")
        print(f"   Bias: {self.bias:.6f}")
        
        return self
    
    def _update_parameters(self, dw, db):
        """
        Actualiza parámetros según el optimizador elegido.
        """
        self.t += 1  # Incrementar contador de tiempo
        
        if self.optimizer == 'gd':
            # Vanilla Gradient Descent
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        
        elif self.optimizer == 'momentum':
            # Gradient Descent con Momentum
            # v = beta * v + (1 - beta) * gradient
            # theta = theta - lr * v
            self.v_w = self.momentum * self.v_w + (1 - self.momentum) * dw
            self.v_b = self.momentum * self.v_b + (1 - self.momentum) * db
            
            self.weights -= self.lr * self.v_w
            self.bias -= self.lr * self.v_b
        
        elif self.optimizer == 'rmsprop':
            # RMSprop: Divide learning rate por raíz del promedio de gradientes al cuadrado
            # s = beta2 * s + (1 - beta2) * gradient^2
            # theta = theta - lr * gradient / sqrt(s + epsilon)
            self.s_w = self.beta2 * self.s_w + (1 - self.beta2) * (dw ** 2)
            self.s_b = self.beta2 * self.s_b + (1 - self.beta2) * (db ** 2)
            
            self.weights -= self.lr * dw / (np.sqrt(self.s_w) + self.epsilon)
            self.bias -= self.lr * db / (np.sqrt(self.s_b) + self.epsilon)
        
        elif self.optimizer == 'adam':
            # Adam: Combina Momentum y RMSprop
            # m = beta1 * m + (1 - beta1) * gradient (primer momento)
            # v = beta2 * v + (1 - beta2) * gradient^2 (segundo momento)
            # m_corrected = m / (1 - beta1^t) (corrección de sesgo)
            # v_corrected = v / (1 - beta2^t)
            # theta = theta - lr * m_corrected / (sqrt(v_corrected) + epsilon)
            
            # Actualizar momentos
            self.v_w = self.momentum * self.v_w + (1 - self.momentum) * dw
            self.v_b = self.momentum * self.v_b + (1 - self.momentum) * db
            self.s_w = self.beta2 * self.s_w + (1 - self.beta2) * (dw ** 2)
            self.s_b = self.beta2 * self.s_b + (1 - self.beta2) * (db ** 2)
            
            # Corrección de sesgo
            v_w_corrected = self.v_w / (1 - self.momentum ** self.t)
            v_b_corrected = self.v_b / (1 - self.momentum ** self.t)
            s_w_corrected = self.s_w / (1 - self.beta2 ** self.t)
            s_b_corrected = self.s_b / (1 - self.beta2 ** self.t)
            
            # Actualización
            self.weights -= self.lr * v_w_corrected / (np.sqrt(s_w_corrected) + self.epsilon)
            self.bias -= self.lr * v_b_corrected / (np.sqrt(s_b_corrected) + self.epsilon)
    
    def predict(self, X):
        """Hace predicciones"""
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        """Calcula R²"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print("✅ Clase GradientDescentOptimizer definida")

### Probemos las diferentes variantes

In [ ]:
# Generar datos de prueba
X_train, X_test, y_train, y_test = generate_synthetic_regression(
    n_samples=1000,
    n_features=1,
    noise=10.0,
    random_state=42
)

print(f"📊 Datos generados: {X_train.shape[0]} ejemplos de entrenamiento")

In [ ]:
# Comparar las tres variantes básicas
variants = [
    ('Batch GD', None),
    ('Mini-batch GD', 32),
    ('SGD', 1)
]

results = {}

for name, batch_size in variants:
    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    
    model = GradientDescentOptimizer(
        learning_rate=0.01,
        n_iterations=100,
        batch_size=batch_size,
        optimizer='gd'
    )
    model.fit(X_train, y_train)
    
    r2 = model.score(X_test, y_test)
    results[name] = {
        'model': model,
        'r2': r2
    }
    print(f"   R² en test: {r2:.4f}")

In [ ]:
# Visualizar convergencia de las tres variantes
fig = go.Figure()

colors = ['blue', 'green', 'red']
for (name, _), color in zip(variants, colors):
    losses = results[name]['model'].losses
    fig.add_trace(go.Scatter(
        y=losses,
        mode='lines',
        name=name,
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title="Comparación de Convergencia: Batch vs Mini-batch vs SGD",
    xaxis_title="Época",
    yaxis_title="Loss (MSE)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Observaciones:")
print("   • Batch GD: Convergencia suave y estable")
print("   • Mini-batch: Balance entre velocidad y estabilidad")
print("   • SGD: Convergencia ruidosa pero puede explorar más")

### Comparación de Optimizadores Avanzados

In [ ]:
# Comparar optimizadores
optimizers = ['gd', 'momentum', 'rmsprop', 'adam']
opt_results = {}

for opt in optimizers:
    print(f"\n🔧 Entrenando con {opt.upper()}...")
    
    model = GradientDescentOptimizer(
        learning_rate=0.01,
        n_iterations=200,
        batch_size=32,
        optimizer=opt
    )
    model.fit(X_train, y_train)
    
    r2 = model.score(X_test, y_test)
    opt_results[opt] = {
        'model': model,
        'r2': r2
    }

# Visualizar
fig = go.Figure()

colors = ['blue', 'orange', 'green', 'red']
for opt, color in zip(optimizers, colors):
    losses = opt_results[opt]['model'].losses
    r2 = opt_results[opt]['r2']
    fig.add_trace(go.Scatter(
        y=losses,
        mode='lines',
        name=f'{opt.upper()} (R²={r2:.4f})',
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title="Comparación de Optimizadores",
    xaxis_title="Época",
    yaxis_title="Loss (MSE)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n📊 Comparación de Optimizadores:")
print("\n" + "="*60)
print(f"{'Optimizador':<15} {'R² Test':<10} {'Loss Final':<15}")
print("="*60)
for opt in optimizers:
    r2 = opt_results[opt]['r2']
    loss = opt_results[opt]['model'].losses[-1]
    print(f"{opt.upper():<15} {r2:<10.4f} {loss:<15.6f}")
print("="*60)

print("\n💡 Adam típicamente converge más rápido y de forma más estable.")

---
## 🏭 5. Versión con Framework (PyTorch/TensorFlow)

Los frameworks modernos implementan estos optimizadores de forma altamente optimizada.

In [ ]:
# Ejemplo con TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras

# Crear modelo simple
model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(1,))
])

# Configurar diferentes optimizadores
tf_optimizers = {
    'SGD': keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.01),
    'Adam': keras.optimizers.Adam(learning_rate=0.01)
}

tf_results = {}

for name, optimizer in tf_optimizers.items():
    print(f"\n🔧 Entrenando con {name}...")
    
    # Compilar modelo
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Entrenar
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=0
    )
    
    # Evaluar
    test_loss = model.evaluate(X_test, y_test, verbose=0)
    tf_results[name] = {
        'history': history.history,
        'test_loss': test_loss
    }
    print(f"   Loss final en test: {test_loss[0]:.6f}")

print("\n✅ Frameworks como TensorFlow/PyTorch ofrecen:")
print("   • Implementaciones altamente optimizadas (C++/CUDA)")
print("   • Diferenciación automática (no calculas gradientes manualmente)")
print("   • Soporte para GPU aceleración")
print("   • Muchos optimizadores adicionales (AdaGrad, AdaDelta, etc.)")

---
## 🎯 6. Ejercicios Prácticos

### 🟢 Ejercicio 1: Efecto del Learning Rate

Experimenta con diferentes learning rates y observa el efecto en la convergencia.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entender el impacto del learning rate
    
    Instrucciones:
    1. Entrena 3 modelos con learning_rates: [0.001, 0.01, 0.1]
    2. Usa batch_size=32, n_iterations=200
    3. Grafica las curvas de loss
    4. Retorna el learning rate que converge más rápido
    
    Returns:
    --------
    best_lr : float
        El learning rate óptimo
    """
    # TODO: Tu código aquí
    # learning_rates = [0.001, 0.01, 0.1]
    # for lr in learning_rates:
    #     model = GradientDescentOptimizer(learning_rate=lr, ...)
    #     ...
    
    pass

# Descomentar para probar
# best_lr = ejercicio_1()
# print(f"\n📊 Mejor learning rate: {best_lr}")

### 🟡 Ejercicio 2: Implementar Early Stopping

El early stopping detiene el entrenamiento cuando la pérdida en validación deja de mejorar.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Implementar early stopping
    
    Instrucciones:
    1. Modifica GradientDescentOptimizer para aceptar un conjunto de validación
    2. En cada época, calcula loss en validación
    3. Si el loss no mejora en 'patience' épocas, detén el entrenamiento
    4. Restaura los mejores pesos encontrados
    
    Pistas:
    - Guarda best_loss y best_weights
    - Usa un contador para patience
    - Compara loss_val con best_loss en cada época
    
    Returns:
    --------
    epoch_stopped : int
        Época en la que se detuvo el entrenamiento
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# epoch = ejercicio_2()
# print(f"\n📊 Entrenamiento detenido en época: {epoch}")

### 🔴 Ejercicio 3: Learning Rate Scheduling

Implementa un schedule que reduzca el learning rate durante el entrenamiento.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Implementar learning rate decay
    
    Instrucciones:
    1. Implementa 3 estrategias de decay:
       a) Step decay: lr = lr * decay_rate cada N épocas
       b) Exponential decay: lr = lr_initial * e^(-decay_rate * epoch)
       c) 1/t decay: lr = lr_initial / (1 + decay_rate * epoch)
    2. Compara las tres estrategias
    3. Grafica learning rate vs época y loss vs época
    
    Returns:
    --------
    dict : {'step': losses, 'exponential': losses, '1/t': losses}
    """
    # TODO: Tu código aquí
    # Pista: Modifica el learning rate en cada época
    # if epoch % step_size == 0:
    #     self.lr *= decay_rate
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("\n📊 Comparación de estrategias de decay completada")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Gradient Descent** es el algoritmo de optimización fundamental en ML, usado para minimizar funciones de pérdida

2. **La regla de actualización** es simple pero poderosa: $\theta := \theta - \alpha \nabla_\theta J(\theta)$

3. **Tres variantes principales**:
   - Batch GD: Usa todos los datos (estable pero lento)
   - SGD: Usa un ejemplo (rápido pero ruidoso)
   - Mini-batch GD: Balance óptimo (más usado en práctica)

4. **El learning rate es crítico**:
   - Muy pequeño: Convergencia muy lenta
   - Muy grande: Oscilaciones o divergencia
   - Típicamente se usa: 0.001, 0.01, 0.1 (probar varios)

5. **Optimizadores avanzados** mejoran la convergencia:
   - **Momentum**: Acelera en direcciones consistentes
   - **RMSprop**: Adapta learning rate por parámetro
   - **Adam**: Combina momentum + RMSprop (más popular)

6. **En producción** usa frameworks (TensorFlow, PyTorch) que implementan estos algoritmos de forma altamente optimizada

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Adam: A Method for Stochastic Optimization"** - Kingma & Ba (2014)
  - [Link al paper](https://arxiv.org/abs/1412.6980)
  - Introduce el optimizador Adam, ahora el más usado en deep learning

- **"On the importance of initialization and momentum in deep learning"** - Sutskever et al. (2013)
  - Explica por qué momentum es tan efectivo

#### 📖 Recursos Interactivos

- **Distill.pub - "Why Momentum Really Works"**
  - [https://distill.pub/2017/momentum/](https://distill.pub/2017/momentum/)
  - Visualizaciones interactivas excelentes

- **CS231n - Optimization Notes**
  - [http://cs231n.github.io/optimization-1/](http://cs231n.github.io/optimization-1/)
  - Explicación detallada con visualizaciones

#### 🎥 Videos Recomendados

- **Andrew Ng - Gradient Descent Lectures**
  - Coursera Machine Learning Course
  - Explicación clara y paso a paso

- **3Blue1Brown - Gradient Descent**
  - Intuición geométrica hermosa

#### 💻 Implementaciones de Referencia

- [PyTorch Optimizers](https://pytorch.org/docs/stable/optim.html)
- [TensorFlow Optimizers](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)
- [JAX Optimizers (Optax)](https://github.com/deepmind/optax)

### 🤔 Preguntas para Reflexionar

1. **¿Por qué Adam funciona tan bien en la práctica?**
   - Combina lo mejor de momentum y adaptive learning rates
   - Funciona bien con valores por defecto

2. **¿Cuándo usar SGD vs Adam?**
   - SGD + momentum puede generalizar mejor (menos overfitting)
   - Adam converge más rápido
   - Depende del problema y arquitectura

3. **¿Cómo escalar el learning rate con el batch size?**
   - Regla común: lr_new = lr_base * (batch_size_new / batch_size_base)
   - Ver "Linear Scaling Rule" de Goyal et al. (2017)

### 📊 Tabla de Referencia Rápida

| Optimizador | Pros | Contras | Cuándo usar |
|-------------|------|---------|-------------|
| **SGD** | Simple, bien entendido | Requiere tunear LR | Cuando tienes tiempo |
| **SGD + Momentum** | Acelera convergencia | Aún requiere tunear | Recomendado para CV |
| **Adam** | Funciona out-of-the-box | Puede overfittear | Primera opción (default) |
| **RMSprop** | Bueno para RNNs | Menos usado ahora | Problemas recurrentes |

---

## ➡️ Próximo Paso

En el siguiente notebook, **03. Regresión Logística**, aplicaremos gradient descent a nuestro primer problema de **clasificación**:

- Pasar de regresión (predecir valores continuos) a clasificación (predecir categorías)
- La función sigmoide para probabilidades
- Cross-entropy como función de pérdida
- Interpretación probabilística
- Métricas de clasificación (accuracy, precision, recall)

---

<div align="center">

**🎉 ¡Has dominado el corazón del Machine Learning! 🎉**

**Continúa con: [03. Regresión Logística](03-regresion-logistica.ipynb)**

[← 01. Regresión Lineal](01-regresion-lineal.ipynb) | [Índice de ML Clásico](README.md) | [03. Regresión Logística →](03-regresion-logistica.ipynb)

</div>

<a name='5'></a>

---
## 🎓 5. Ejercicios Prácticos Guiados (100 pts)

Esta sección contiene **8 ejercicios autogradeados** que te guiarán paso a paso en la implementación de gradient descent y sus optimizadores avanzados.

### 📊 Sistema de Puntos

| Ejercicio | Función | Puntos | Dificultad | Tiempo |
|-----------|---------|--------|------------|--------|
| 1 | `compute_gradient` | 10 | 🟢 Fácil | 8 min |
| 2 | `gradient_descent_step` | 10 | 🟢 Fácil | 8 min |
| 3 | `batch_gradient_descent` | 15 | 🟡 Medio | 12 min |
| 4 | `create_mini_batches` | 10 | 🟢 Fácil | 8 min |
| 5 | `sgd_with_momentum` | 15 | 🟡 Medio | 12 min |
| 6 | `rmsprop_update` | 15 | 🟡 Medio | 12 min |
| 7 | `adam_optimizer` | 20 | 🔴 Difícil | 15 min |
| 8 | `learning_rate_decay` | 5 | 🟢 Fácil | 5 min |
| **TOTAL** | | **100** | | **80 min** |

**Mínimo para aprobar:** 70 puntos

### 🎯 Instrucciones

1. **Lee el docstring** de cada función cuidadosamente
2. **Implementa el código** entre `# YOUR CODE STARTS HERE` y `# YOUR CODE ENDS HERE`
3. **Ejecuta la celda de test** inmediatamente después de cada función
4. **Verifica los resultados** - el autograder te dirá si pasaste todos los tests
5. **Obtén al menos 70 puntos** para aprobar el notebook

### 🚀 Importar el Autograder

In [ ]:
# Importar sistema de autograding
import sys
sys.path.append('./tests')
from test_02_gradient_descent import GradientDescentGrader

# Inicializar grader
grader = GradientDescentGrader()

print("✅ Autograder importado correctamente")
print(f"📊 Puntos totales disponibles: {grader.total_points}")
print(f"📊 Puntos mínimos para aprobar: 70")
print("\n🎯 ¡Comencemos con los ejercicios!")

<a name='ex-1'></a>

---
### Exercise 1 - compute_gradient

**GRADED FUNCTION: `compute_gradient`**

**Dificultad:** 🟢 Fácil | **Puntos:** 10 | **Tiempo estimado:** 8 min

#### Objetivo

Implementa la función que calcula el gradiente de la función de costo MSE con respecto a los pesos y el bias.

#### Teoría

Para la función de costo MSE:

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$

Las derivadas parciales son:

$$\frac{\partial J}{\partial w} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x^{(i)}$$

$$\frac{\partial J}{\partial b} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

Donde $\hat{y}^{(i)} = w \cdot x^{(i)} + b$

#### Instrucciones

Implementa `compute_gradient(X, y, w, b)` que retorna `(dw, db)`:
1. Calcula las predicciones $\hat{y}$
2. Calcula el error $e = \hat{y} - y$
3. Calcula el gradiente de los pesos: `dw`
4. Calcula el gradiente del bias: `db`

**Nota:** Puedes omitir el factor 2 (solo escala el learning rate).

### Ejercicio: Implementar función básica (15 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_1():
    """Implementar función básica"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint1
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_1...')
try:
    # result = exercise_1(...)
    print('Expected: ')
    print('✅ +15 pts')
except: print('❌ Error')


### Ejercicio: Calcular métrica (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_2():
    """Calcular métrica"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_2...')
try:
    # result = exercise_2(...)
    print('Expected: ')
    print('✅ +20 pts')
except: print('❌ Error')


### Ejercicio: Entrenar modelo (30 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_3():
    """Entrenar modelo"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_3...')
try:
    # result = exercise_3(...)
    print('Expected: ')
    print('✅ +30 pts')
except: print('❌ Error')


### Ejercicio: Predicción (15 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_4():
    """Predicción"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_4...')
try:
    # result = exercise_4(...)
    print('Expected: ')
    print('✅ +15 pts')
except: print('❌ Error')


### Ejercicio: Evaluación (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_5():
    """Evaluación"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_5...')
try:
    # result = exercise_5(...)
    print('Expected: ')
    print('✅ +20 pts')
except: print('❌ Error')
